# 08. Attention core — MHA/MQA/GQA, MLA, and DeepSeek-V4 compressed attention

This notebook preserves the computation graph and reduces only tensor sizes.

1. MHA/MQA/GQA cache geometry
2. DeepSeek-V2 style MLA with latent KV cache and absorption
3. DeepSeek-V4 style sequence-dimension compression
4. CSA: compressed sparse attention with learned indexing and sparse fine attention
5. HCA: heavier compression followed by dense attention
6. a hybrid CSA/HCA layer


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## 1. MHA, MQA, GQA cache geometry


In [ ]:
batch = 2
length = 16
head_dim = 8
query_heads = 4

mha_k = torch.randn(batch, query_heads, length, head_dim, device=device)
mqa_k = torch.randn(batch, 1, length, head_dim, device=device)
gqa_k = torch.randn(batch, 2, length, head_dim, device=device)

print("MHA K elements:", mha_k.numel())
print("MQA K elements:", mqa_k.numel())
print("GQA K elements:", gqa_k.numel())


## 2. RoPE helper


In [ ]:
def apply_rope(x, positions):
    dim = x.size(-1)
    assert dim % 2 == 0

    index = torch.arange(0, dim, 2, device=x.device, dtype=torch.float32)
    inverse_frequency = 1.0 / (10000 ** (index / dim))
    angle = positions.float()[:, None] * inverse_frequency[None]

    cos = angle.cos()[None, None]
    sin = angle.sin()[None, None]

    even = x[..., 0::2]
    odd = x[..., 1::2]
    rotated_even = even * cos - odd * sin
    rotated_odd = even * sin + odd * cos
    return torch.stack([rotated_even, rotated_odd], dim=-1).flatten(-2)


## 3. MLA: low-rank Q/KV, decoupled RoPE, causal attention, latent cache


In [ ]:
class TinyMLA(nn.Module):
    def __init__(
        self,
        model_dim=32,
        heads=4,
        q_rank=12,
        kv_rank=8,
        nope_dim=6,
        rope_dim=2,
        value_dim=6,
    ):
        super().__init__()
        self.heads = heads
        self.kv_rank = kv_rank
        self.nope_dim = nope_dim
        self.rope_dim = rope_dim
        self.value_dim = value_dim

        self.q_down = nn.Linear(model_dim, q_rank, bias=False)
        self.q_norm = nn.RMSNorm(q_rank)
        self.q_up = nn.Linear(
            q_rank,
            heads * (nope_dim + rope_dim),
            bias=False,
        )

        self.kv_down = nn.Linear(
            model_dim,
            kv_rank + rope_dim,
            bias=False,
        )
        self.kv_norm = nn.RMSNorm(kv_rank)
        self.kv_up = nn.Linear(
            kv_rank,
            heads * (nope_dim + value_dim),
            bias=False,
        )
        self.out = nn.Linear(heads * value_dim, model_dim, bias=False)

    def forward(self, hidden):
        batch, length, _ = hidden.shape
        positions = torch.arange(length, device=hidden.device)

        q_latent = self.q_norm(self.q_down(hidden))
        q = self.q_up(q_latent).view(
            batch,
            length,
            self.heads,
            self.nope_dim + self.rope_dim,
        ).transpose(1, 2)
        q_nope, q_rope = q.split([self.nope_dim, self.rope_dim], dim=-1)

        compressed = self.kv_down(hidden)
        kv_latent, shared_rope = compressed.split(
            [self.kv_rank, self.rope_dim],
            dim=-1,
        )
        normalized_kv = self.kv_norm(kv_latent)

        kv = self.kv_up(normalized_kv).view(
            batch,
            length,
            self.heads,
            self.nope_dim + self.value_dim,
        ).transpose(1, 2)
        k_nope, value = kv.split(
            [self.nope_dim, self.value_dim],
            dim=-1,
        )

        k_rope = shared_rope[:, None].expand(
            -1,
            self.heads,
            -1,
            -1,
        )
        query = torch.cat(
            [q_nope, apply_rope(q_rope, positions)],
            dim=-1,
        )
        key = torch.cat(
            [k_nope, apply_rope(k_rope, positions)],
            dim=-1,
        )

        attended = F.scaled_dot_product_attention(
            query,
            key,
            value,
            is_causal=True,
        )
        attended = attended.transpose(1, 2).contiguous().flatten(2)

        diagnostics = {
            "q_latent": q_latent,
            "kv_latent": kv_latent,
            "shared_rope": shared_rope,
        }
        return self.out(attended), diagnostics


mla = TinyMLA().to(device)
hidden = torch.randn(2, 12, 32, device=device)
mla_output, mla_diag = mla(hidden)

full_cache = hidden.size(0) * hidden.size(1) * 4 * (8 + 6)
latent_cache = mla_diag["kv_latent"].numel() + mla_diag["shared_rope"].numel()

print("MLA output:", mla_output.shape)
print("full-like cache elements:", full_cache)
print("latent cache elements:", latent_cache)


## 4. Sequence compression used by V4-style long-context attention

The compression axis is sequence length, not merely head/channel rank.
Overlapping windows produce fewer learned compressed KV entries.


In [ ]:
class SequenceCompressor(nn.Module):
    def __init__(self, model_dim=32, compressed_dim=16, window=4, stride=2):
        super().__init__()
        self.window = window
        self.stride = stride

        self.score = nn.Linear(model_dim, 1)
        self.key = nn.Linear(model_dim, compressed_dim, bias=False)
        self.value = nn.Linear(model_dim, compressed_dim, bias=False)

    def forward(self, hidden):
        windows = hidden.unfold(
            dimension=1,
            size=self.window,
            step=self.stride,
        )
        windows = windows.permute(0, 1, 3, 2).contiguous()

        weights = self.score(windows).squeeze(-1).softmax(dim=-1)
        pooled = torch.einsum(
            "bnw,bnwd->bnd",
            weights,
            windows,
        )
        return self.key(pooled), self.value(pooled), pooled


compressor = SequenceCompressor().to(device)
compressed_k, compressed_v, compressed_summary = compressor(hidden)
print("token length:", hidden.size(1))
print("compressed length:", compressed_summary.size(1))


## 5. CSA — compressed sparse attention

A cheap learned indexer scores compressed entries.
Only top-k compressed entries are gathered for the expensive fine attention;
we do not calculate a full token-by-token QK matrix and mask it afterwards.


In [ ]:
class CompressedSparseAttention(nn.Module):
    def __init__(
        self,
        model_dim=32,
        compressed_dim=16,
        heads=4,
        top_k=3,
        window=4,
        stride=2,
    ):
        super().__init__()
        self.heads = heads
        self.head_dim = compressed_dim // heads
        self.top_k = top_k

        self.compressor = SequenceCompressor(
            model_dim,
            compressed_dim,
            window,
            stride,
        )
        self.query = nn.Linear(model_dim, compressed_dim, bias=False)
        self.index_query = nn.Linear(model_dim, compressed_dim, bias=False)
        self.index_key = nn.Linear(model_dim, compressed_dim, bias=False)
        self.out = nn.Linear(compressed_dim, model_dim, bias=False)

    def forward(self, hidden):
        batch, length, _ = hidden.shape
        compressed_k, compressed_v, summary = self.compressor(hidden)

        index_q = F.normalize(self.index_query(hidden), dim=-1)
        index_k = F.normalize(self.index_key(summary), dim=-1)
        index_scores = torch.einsum(
            "btd,bnd->btn",
            index_q,
            index_k,
        )

        selected_scores, selected_ids = index_scores.topk(
            k=min(self.top_k, summary.size(1)),
            dim=-1,
        )

        batch_ids = torch.arange(batch, device=hidden.device)[:, None, None]
        selected_k = compressed_k[batch_ids, selected_ids]
        selected_v = compressed_v[batch_ids, selected_ids]

        query = self.query(hidden).view(
            batch,
            length,
            self.heads,
            self.head_dim,
        )
        selected_k = selected_k.view(
            batch,
            length,
            -1,
            self.heads,
            self.head_dim,
        )
        selected_v = selected_v.view(
            batch,
            length,
            -1,
            self.heads,
            self.head_dim,
        )

        fine_scores = torch.einsum(
            "bthd,btkhd->bthk",
            query,
            selected_k,
        ) / math.sqrt(self.head_dim)
        fine_weights = fine_scores.softmax(dim=-1)

        attended = torch.einsum(
            "bthk,btkhd->bthd",
            fine_weights,
            selected_v,
        ).reshape(batch, length, -1)

        diagnostics = {
            "selected_ids": selected_ids,
            "index_scores": selected_scores,
            "fine_score_count": fine_scores.numel(),
        }
        return self.out(attended), diagnostics


csa = CompressedSparseAttention().to(device)
csa_output, csa_diag = csa(hidden)

dense_token_scores = (
    hidden.size(0) * hidden.size(1) * hidden.size(1) * 4
)
print("CSA output:", csa_output.shape)
print("selected compressed ids:", csa_diag["selected_ids"][0, :3])
print("fine scores computed:", csa_diag["fine_score_count"])
print("dense token-head scores would be:", dense_token_scores)


## 6. HCA — heavier compression, then dense attention

HCA uses fewer sequence entries than CSA. Since that compressed sequence is
short, attention over **all** heavily-compressed entries is feasible.


In [ ]:
class HeavilyCompressedAttention(nn.Module):
    def __init__(
        self,
        model_dim=32,
        compressed_dim=16,
        heads=4,
    ):
        super().__init__()
        self.heads = heads
        self.head_dim = compressed_dim // heads

        self.compressor = SequenceCompressor(
            model_dim,
            compressed_dim,
            window=6,
            stride=4,
        )
        self.query = nn.Linear(model_dim, compressed_dim, bias=False)
        self.out = nn.Linear(compressed_dim, model_dim, bias=False)

    def forward(self, hidden):
        batch, length, _ = hidden.shape
        compressed_k, compressed_v, summary = self.compressor(hidden)

        query = self.query(hidden).view(
            batch,
            length,
            self.heads,
            self.head_dim,
        ).transpose(1, 2)
        key = compressed_k.view(
            batch,
            -1,
            self.heads,
            self.head_dim,
        ).transpose(1, 2)
        value = compressed_v.view(
            batch,
            -1,
            self.heads,
            self.head_dim,
        ).transpose(1, 2)

        attended = F.scaled_dot_product_attention(query, key, value)
        attended = attended.transpose(1, 2).contiguous().flatten(2)

        return self.out(attended), summary


hca = HeavilyCompressedAttention().to(device)
hca_output, hca_summary = hca(hidden)
print("HCA output:", hca_output.shape)
print("HCA compressed length:", hca_summary.size(1))


## 7. Hybrid CSA/HCA layer


In [ ]:
class V4HybridAttentionLayer(nn.Module):
    def __init__(self, model_dim=32):
        super().__init__()
        self.norm_csa = nn.RMSNorm(model_dim)
        self.norm_hca = nn.RMSNorm(model_dim)
        self.csa = CompressedSparseAttention(model_dim=model_dim)
        self.hca = HeavilyCompressedAttention(model_dim=model_dim)
        self.mix = nn.Linear(2 * model_dim, model_dim, bias=False)

    def forward(self, hidden):
        csa_output, csa_diag = self.csa(self.norm_csa(hidden))
        hca_output, hca_summary = self.hca(self.norm_hca(hidden))

        mixed = self.mix(torch.cat([csa_output, hca_output], dim=-1))
        return hidden + mixed, csa_diag, hca_summary


v4_layer = V4HybridAttentionLayer().to(device)
v4_output, csa_diag, hca_summary = v4_layer(hidden)
loss = v4_output.square().mean()
loss.backward()

print("hybrid output:", v4_output.shape)
print("CSA compressed selection:", csa_diag["selected_ids"].shape)
print("HCA compressed sequence:", hca_summary.shape)
print("compressor grad:",
      v4_layer.csa.compressor.score.weight.grad.norm().item())


## References and provenance

- **MLA**: low-rank latent KV cache, decoupled positional subspace, causal attention.
- **DeepSeek-V4**: official model card states that V4 combines **CSA** and **HCA**,
  where CSA compresses KV along the sequence dimension and performs sparse
  attention, while HCA applies heavier sequence compression followed by dense
  attention.
- The notebook reproduces that disclosed computation structure at small scale.
  Large production kernels and undisclosed deployment constants are not
  substituted with unrelated algorithms.
